# ND-AWS at p=1, n = 100 — one arm per run

Runs one arm of the 100-qubit ND-AWS study: noise-directed adaptive remapping (NDAR) over a
p=1 warm-started QAOA sampler, ten Hamiltonian instances, three restarts each. Six arms are
available — three Hamiltonian classes (ER-10, ER-20, RG-3) on 100 qubits of the Heron QPU,
and the three matching noiseless matrix-product-state references. The ansatz is selected
separately: `ND` applies the gauge transformations NDAR proposes, `Standard` keeps the
warm start fixed.

Everything the user is meant to change lives in the configuration cell below. The arm table
then fixes the Hamiltonian class, the phase-separator fraction, the backend and the
compilation path.

**Writes**: one experiment set per (instance, restart) into the standardized result store,
all grouped under a single dataset name built from the arm, the ansatz and `RUN_ID`. The
store is `DEFAULT_STORAGE_DIRECTORY`; with that variable unset or blank it is `output/` at
the repository root, no matter which folder the notebook was started from.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
from datetime import datetime

import pandas as pd
import qiskit
from dotenv import load_dotenv

import quapopt
from quapopt import ancillary_functions as anf
from quapopt.ancillary_functions.presets import (generate_random_hamiltonian,
                                                 get_standard_subfolders_hierarchy_full,
                                                 ClassicalHamiltonian,
                                                 get_hamiltonian_class_description)
from quapopt.ancillary_functions.presets.ndar import (build_local_sampler_callable_and_updater_NDAR,
                                                      build_universal_logging_callable,
                                                      LocalSamplerName)
from quapopt.ancillary_functions.presets.wrapped_optimization import get_classical_optimizer_p1_default
from quapopt.circuits import backend_utilities as bck_utils
from quapopt.circuits.backend_utilities.qiskit.qiskit_config import (DEFAULT_QPU_SAMPLER_KWARGS,
                                                                     DEFAULT_SIMULATOR_BACKEND_KWARGS)
from quapopt.circuits.gates.logical.LogicalGateBuilderQiskit import LogicalGateBuilderQiskit
from quapopt.circuits.gates.native.NativeGateBuilderHeron import NativeGateBuilderHeron
from quapopt.data_analysis.data_handling import (HamiltonianModels,
                                                 ResultsLogger,
                                                 STANDARD_NAMES_DATA_TYPES as SNDT,
                                                 STANDARD_NAMES_VARIABLES as SNV)
from quapopt.data_analysis.data_handling.io_utilities.dataset_management import read_dataset_info
from quapopt.meta_algorithms.NDAR import ConvergenceCriterion, ConvergenceCriterionNames
from quapopt.meta_algorithms.NDAR.NDARRunner import NDARRunner
from quapopt.optimization.QAOA import InitialStateType, MixerType, PhaseSeparatorType, QubitMappingType

## Configuration

The only cell meant to be edited.

`ARM` picks one row of the arm table below. `ANSATZ` picks the ansatz. `RUN_ID` is the
suffix of the dataset name: a paired ND / Standard study must use **one** `RUN_ID` on both
arms, so the two datasets differ only by the `no-gauges-version` token. There is no
auto-suffix — a `RUN_ID` that already holds data stops the notebook before anything runs.

The three budget knobs default to the published values: ten instances (the seed of instance
*i* is *i*), three restarts per instance — the shape of the published datasets, about 30
experiment sets each — and 10,000 shots per circuit.

`add_hdls` chooses whether to add local post-processing step between iterations: a
Hamming-distance local search over the neighbourhood of each sample. The paper calls its
distance-2 case Hamming Distance Quadratic Search (HDQS), and `add_hdls` is what switches
it on. The main-text 100-qubit results have it on; the appendix repeats the study with it
off.


In [ ]:
ARM = 'MPS-RG3'
# 'QPU-ER10' | 'QPU-ER20' | 'QPU-RG3' | 'MPS-ER10' | 'MPS-ER20' | 'MPS-RG3'

ANSATZ = 'ND'
# 'ND' | 'Standard'

RUN_ID = 'run-01'

hamiltonian_indices_range = list(range(0, 10))
restarts_range = range(0, 3)
number_of_samples_qiskit = 10 ** 4

#Whether to add Hamming distance local search at each step, 2-local. In the paper, we present results with this on (main) and off (appendix)
add_hdls = True

## Arm table

| ARM | class | instance kwargs | phase-separator fraction | backend | compilation |
|---|---|---|---|---|---|
| QPU-ER10 | Erdos-Renyi, p = 0.1 | `{'p_or_M': 0.1}` | 0.25 | `ibm_boston` | Heron native gates, SABRE routing |
| QPU-ER20 | Erdos-Renyi, p = 0.2 | `{'p_or_M': 0.2}` | 0.10 | `ibm_boston` | Heron native gates, SABRE routing |
| QPU-RG3 | 3-regular | `{'graph_degree': 3}` | 0.80 | `ibm_boston` | Heron native gates, SABRE routing |
| MPS-ER10 | Erdos-Renyi, p = 0.1 | `{'p_or_M': 0.1}` | 1.0 | `aer`, MPS, chi = 40 | logical gates, all-to-all |
| MPS-ER20 | Erdos-Renyi, p = 0.2 | `{'p_or_M': 0.2}` | 1.0 | `aer`, MPS, chi = 40 | logical gates, all-to-all |
| MPS-RG3 | 3-regular | `{'graph_degree': 3}` | 1.0 | `aer`, MPS, chi = 40 | logical gates, all-to-all |

The phase-separator fraction (`time_block_size`) is the share of the largest-magnitude
interactions that one p=1 phase-separator layer implements. The QPU arms use a fraction
below 1 because the routed circuit depth grows with it; the noiseless MPS reference
implements the whole Hamiltonian.

`ANSATZ` selects the initial state: `ws_qaoa_identical` for ND, `ws_qaoa_general` for
Standard. `no_gauges_version` follows from it and is the token that separates the two
datasets.

In [ ]:
ARM_TABLE = {
    'QPU-ER10': {'hamiltonian_model': HamiltonianModels.ErdosRenyi,
                 'instance_kwargs': {'p_or_M': 0.1},
                 'time_block_size': 0.25,
                 'backend_name': 'ibm_boston',
                 'simulation': False, 'noiseless_simulation': False, 'mps_simulation': False,
                 'max_bond_dimension': None},
    'QPU-ER20': {'hamiltonian_model': HamiltonianModels.ErdosRenyi,
                 'instance_kwargs': {'p_or_M': 0.2},
                 'time_block_size': 0.10,
                 'backend_name': 'ibm_boston',
                 'simulation': False, 'noiseless_simulation': False, 'mps_simulation': False,
                 'max_bond_dimension': None},
    'QPU-RG3': {'hamiltonian_model': HamiltonianModels.RegularGraph,
                'instance_kwargs': {'graph_degree': 3},
                'time_block_size': 0.80,
                'backend_name': 'ibm_boston',
                'simulation': False, 'noiseless_simulation': False, 'mps_simulation': False,
                'max_bond_dimension': None},
    'MPS-ER10': {'hamiltonian_model': HamiltonianModels.ErdosRenyi,
                 'instance_kwargs': {'p_or_M': 0.1},
                 'time_block_size': 1.0,
                 'backend_name': 'aer',
                 'simulation': True, 'noiseless_simulation': True, 'mps_simulation': True,
                 'max_bond_dimension': 40},
    'MPS-ER20': {'hamiltonian_model': HamiltonianModels.ErdosRenyi,
                 'instance_kwargs': {'p_or_M': 0.2},
                 'time_block_size': 1.0,
                 'backend_name': 'aer',
                 'simulation': True, 'noiseless_simulation': True, 'mps_simulation': True,
                 'max_bond_dimension': 40},
    'MPS-RG3': {'hamiltonian_model': HamiltonianModels.RegularGraph,
                'instance_kwargs': {'graph_degree': 3},
                'time_block_size': 1.0,
                'backend_name': 'aer',
                'simulation': True, 'noiseless_simulation': True, 'mps_simulation': True,
                'max_bond_dimension': 40},
}

if ARM not in ARM_TABLE:
    raise ValueError(f"unknown ARM {ARM!r}; choose one of {sorted(ARM_TABLE)}")
if ANSATZ not in ('ND', 'Standard'):
    raise ValueError(f"unknown ANSATZ {ANSATZ!r}; choose 'ND' or 'Standard'")

_arm = ARM_TABLE[ARM]
hamiltonian_model = _arm['hamiltonian_model']
instance_kwargs = _arm['instance_kwargs']
time_block_size = _arm['time_block_size']
backend_name = _arm['backend_name']
simulation = _arm['simulation']
noiseless_simulation = _arm['noiseless_simulation']
mps_simulation = _arm['mps_simulation']
max_bond_dimension = _arm['max_bond_dimension']

number_of_qubits = 100
localities = (2,)

if simulation:
    qubit_mapping_type = QubitMappingType.fully_connected
    gate_builder = LogicalGateBuilderQiskit()
else:
    qubit_mapping_type = QubitMappingType.sabre
    gate_builder = NativeGateBuilderHeron()

initial_state_type = (InitialStateType.ws_qaoa_identical if ANSATZ == 'ND'
                      else InitialStateType.ws_qaoa_general)
no_gauges_version = initial_state_type == InitialStateType.ws_qaoa_general

print('arm:', ARM, '| ansatz:', ANSATZ, '| no gauges:', no_gauges_version)

## Credentials

The QPU arms need an IBM account; the MPS arms never touch one. `load_dotenv()` reads a
local `.env` if present, and the check below fails with a readable message instead of
letting the provider constructor raise deep inside Qiskit.

In [ ]:
load_dotenv()
HAVE_IBM = any(os.getenv(k) for k in ('IBM_TOKEN', 'IBM_CREDENTIALS_PATH', 'IBM_ACCOUNT_NAME'))

if simulation:
    provider_ibm = None
else:
    if not HAVE_IBM:
        raise RuntimeError("IBM credentials absent: set IBM_TOKEN or "
                           "IBM_CREDENTIALS_PATH/IBM_ACCOUNT_NAME in the environment or .env; "
                           "the MPS arms run without them")
    provider_ibm = bck_utils.get_qiskit_provider()

qiskit_backend = bck_utils.get_qiskit_backend(backend_name=backend_name,
                                              qiskit_provider=provider_ibm)
print('backend:', qiskit_backend.name)



## Hamiltonian class, storage hierarchy and sampler options

In [ ]:
# None: at 100 qubits the Hamiltonian instances are built on the GPU backend (cupy) when
# cupy is installed and a GPU is visible, and on numpy otherwise.
default_backend = None

ham_class_description = get_hamiltonian_class_description(hamiltonian_model=hamiltonian_model,
                                                          localities=localities, )
experiments_folders_hierarchy = get_standard_subfolders_hierarchy_full(experiment_category='NDAR',
                                                                       experiment_subcategory='WS-QAOA-p1',
                                                                       backend_name=backend_name,
                                                                       simulation=simulation,
                                                                       noiseless_simulation=noiseless_simulation,
                                                                       hamiltonian_class_description=ham_class_description)

if simulation:
    if qiskit_backend.name[0:3] == 'aer':
        qiskit_sampler_options = DEFAULT_SIMULATOR_BACKEND_KWARGS.copy()
    else:
        raise ValueError(f"Backend {qiskit_backend.name} not recognized for simulation. ")

    if mps_simulation:
        qiskit_sampler_options['method'] = 'matrix_product_state'
        qiskit_sampler_options['matrix_product_state_max_bond_dimension'] = max_bond_dimension
else:
    qiskit_sampler_options = DEFAULT_QPU_SAMPLER_KWARGS.copy()

print(default_backend, ham_class_description)

## Ansatz, angle optimizer and the transpiler seed

Angles are set offline: for each NDAR iteration the sampler optimizes gamma on a
reduced-density-matrix simulator with COBYQA plus basinhopping (200 function calls), and
finds the best beta for each gamma internally (`efficient_finding_of_best_beta`), so only
gamma is handed to the optimizer.

The published runs found that beta by a scan over a grid of `betas_search_space_size`
values. `analytical_betas = False` keeps that scan, so this notebook runs the published
configuration. The library can now also find that beta exactly, in closed form, when the
warm-start bias is the same on every qubit, as it is here; `analytical_betas = None`
selects it.

Basinhopping takes random displacement steps, so the search has a seed of its own,
`optimizer_seed`. `None` keeps the seed the optimizer preset configures, which is what makes
a re-run return the same angles; an integer overrides it.

`time_block_seed` selects the interactions the phase-separator layer implements: `-1` takes
the largest-magnitude subset, any other integer shuffles the term list with that seed. The
value set here is only a fallback — the sampler takes a fresh seed on every NDAR iteration
from `time_block_seed_callable` two cells below, so the fallback never reaches a circuit in
this notebook. The seed has no effect at all when `time_block_size = 1.0`: the layer then
holds every interaction and the term order is left alone.

Transpiler seed: pinned to `[42]`, the seed every published circuit was compiled with.

In [ ]:
phase_separator_type = PhaseSeparatorType.QAOA
mixer_type = MixerType.ws_qaoa_identical

# Fallback only: the kwargs updater supplies a seed on every NDAR iteration, so this value
# is what a caller that passes no schedule would get. See time_block_seed_callable below.
time_block_seed = -1
efficient_finding_of_best_beta = True
betas_search_space_size = 10 ** 6
# False: find beta by the grid scan over betas_search_space_size values, as the published
# runs did. None: the closed-form beta, where it applies.
analytical_betas = False
number_of_function_calls_simulator = 200
r_best_results = 1
pass_manager_seeds_list = [42]

# The seed of the offline angle search itself. None keeps the seed the optimizer preset
# configures, which is what makes the search reproducible; an integer overrides it. The
# effective value is in optimizer_metadata['basinhopping_kwargs'] either way.
optimizer_seed = None

classical_optimizer, optimizer_metadata = get_classical_optimizer_p1_default(
    efficient_finding_of_best_beta=efficient_finding_of_best_beta,
    number_of_qubits=number_of_qubits)

optimizer_metadata['betas_search_space_size'] = betas_search_space_size
optimizer_metadata['analytical_betas'] = analytical_betas
optimizer_metadata['optimizer_seed'] = optimizer_seed

In [ ]:
ansatz_config_dict = {'time_block_size': time_block_size,
                      'time_block_seed': time_block_seed if isinstance(time_block_seed, int) else None,
                      'phase_separator_type': phase_separator_type,
                      'mixer_type': mixer_type,
                      'initial_state_type': initial_state_type,
                      }

pass_manager_kwargs_main = {'optimization_level': 3 if qubit_mapping_type == QubitMappingType.sabre else 0}

qiskit_config_dict_main = {'qubit_mapping_type': qubit_mapping_type,
                           'qiskit_backend': qiskit_backend,
                           'backend_name': backend_name,
                           'pass_manager_seeds_list': pass_manager_seeds_list,
                           'gate_builder': gate_builder,
                           'qubit_indices_physical': None,
                           'simulation': simulation,
                           'noiseless_simulation': noiseless_simulation,
                           'mps_simulation': mps_simulation,
                           'max_bond_dimension': max_bond_dimension,
                           'no_gauges_version': no_gauges_version,
                           'qiskit_sampler_options': qiskit_sampler_options,
                           'add_hdls': add_hdls,
                           }

qiskit_metadata_additional_main = {'qiskit_version': qiskit.__version__,
                                   'backend_name': backend_name,
                                   'simulation': simulation,
                                   'noiseless_simulation': noiseless_simulation,
                                   'gate_builder_class': gate_builder.__class__.__name__,
                                   'pass_manager_seeds_list': pass_manager_seeds_list,
                                   }

simulator_config_dict = {'classical_optimizer': classical_optimizer,
                         'betas_search_space_size': betas_search_space_size,
                         'analytical_betas': analytical_betas,
                         'efficient_finding_of_best_beta': efficient_finding_of_best_beta,
                         'optimizer_seed': optimizer_seed,
                         }

## Bias schedule and phase-separator subset

The bias `c` is the per-qubit weight of the warm-started initial product state: each qubit
is prepared with probability `c` of being measured against its warm-start value, through a
rotation by `2 arcsin(sqrt(c))`. So `c = 0.5` is the unbiased |+> state and a smaller `c`
sits closer to the warm start. Both arms of this notebook use the published schedule:
`[0.5]` at iteration 0, `[0.1, 0.05]` for iterations 1 to 3, `[0.05, 0.025]` from iteration
4 on. A list means both values are tried and the better one is kept.

`time_block_seed_callable` selects which interactions enter the phase-separator layer:
iterations 0 to 3 take the largest-magnitude subset (`-1`), and every later iteration takes
a fresh random subset drawn with the iteration index as seed. The ND and Standard arms stay
comparable because one seed selects the same subset of interaction terms in both; only the
signs of those terms differ, through the bitflip gauges.

Because the seed changes with the iteration, the metadata row records the schedule.

In [ ]:
def bias_list_iteration_callable(ndar_iteration_index):
    if ndar_iteration_index == 0:
        return [0.5]
    elif ndar_iteration_index <= 3:
        return [0.1, 0.05]
    else:
        return [0.05, 0.025]


# From this iteration on, the phase-separator subset is drawn at random with the iteration
# index as seed; before it, -1 takes the largest-magnitude subset.
time_block_seed_random_from_iteration = 4


def time_block_seed_callable(ndar_iteration_index):
    if ndar_iteration_index < time_block_seed_random_from_iteration:
        return -1
    else:
        return ndar_iteration_index


# What the metadata row records, built from the same constant the callable uses so the two
# cannot drift apart.
if time_block_size == 1.0:
    time_block_seed_schedule = ('unused: at time_block_size=1.0 the phase separator holds '
                                'every interaction and the seed does not reach the circuit')
else:
    time_block_seed_schedule = (f'-1 for iterations 0 to {time_block_seed_random_from_iteration - 1}, '
                                f'then the NDAR iteration index')

config_qiskit_sampler_main = {'r_best_results': r_best_results,
                              'number_of_function_calls_simulator': number_of_function_calls_simulator,
                              'number_of_samples_qiskit': number_of_samples_qiskit,
                              'optimize_bias_offline': False,
                              'bias_list_iteration_callable': bias_list_iteration_callable,
                              'time_block_seed_callable': time_block_seed_callable,
                              'time_block_seed_schedule': time_block_seed_schedule,
                              'ansatz_config': ansatz_config_dict,
                              'simulator_config': simulator_config_dict,
                              }

## Dataset name, termination rule and the name guard

The dataset name is assembled from the arm: `QPU_` or `simulation_noiseless_mps_chi=40_`,
the phase-separator fraction when it is not 1.0, the Hamiltonian class, `no-gauges-version_`
for the Standard ansatz, `HDLS_` for the local-search refinement, and `RUN_ID`. The six arms
therefore produce

| ARM | ND | Standard |
|---|---|---|
| QPU-ER10 | `NDAWS_QPU_k=0.25_ER-0.1_HDLS_<RUN_ID>` | `NDAWS_QPU_k=0.25_ER-0.1_no-gauges-version_HDLS_<RUN_ID>` |
| QPU-ER20 | `NDAWS_QPU_k=0.1_ER-0.2_HDLS_<RUN_ID>` | `NDAWS_QPU_k=0.1_ER-0.2_no-gauges-version_HDLS_<RUN_ID>` |
| QPU-RG3 | `NDAWS_QPU_k=0.8_RG-3_HDLS_<RUN_ID>` | `NDAWS_QPU_k=0.8_RG-3_no-gauges-version_HDLS_<RUN_ID>` |
| MPS-ER10 | `NDAWS_simulation_noiseless_mps_chi=40_ER-0.1_HDLS_<RUN_ID>` | same, with `no-gauges-version_` |
| MPS-ER20 | `NDAWS_simulation_noiseless_mps_chi=40_ER-0.2_HDLS_<RUN_ID>` | same, with `no-gauges-version_` |
| MPS-RG3 | `NDAWS_simulation_noiseless_mps_chi=40_RG-3_HDLS_<RUN_ID>` | same, with `no-gauges-version_` |

The name is not a folder: it is the key the store registers experiment sets under, one
`DSN=<dataset name>.csv` index file per dataset, which is what the analysis notebooks read
back. The token there is the code's `HDLS_`, the paper's HDQS.

The guard below refuses a dataset name that already holds experiment sets. Mixing two runs
under one name would make the analyzer's best-of-all-sets selection silently span both.

Termination: NDAR stops when the best cost has not improved for three consecutive
iterations. The known-optimum break is off in every arm here — the run cell passes
`break_after_finding_ground_state=simulation and not mps_simulation`, which no row of the
arm table satisfies — so a run that samples the optimum still spends its three patience
iterations. The 20-qubit study in
[01a](01a_ndaws_amplitude_damping_run.ipynb) turns that break on.

In [ ]:
convergence_value = 3
convergence_criterion = ConvergenceCriterion(convergence_criterion_name=ConvergenceCriterionNames.MaxUnsuccessfulTrials,
                                             convergence_value=convergence_value)

local_samplers = [LocalSamplerName.QiskitWithOfflineAnglesP1]

dataset_name = "NDAWS_"

if simulation:
    dataset_name += 'simulation_'
    if noiseless_simulation:
        dataset_name += 'noiseless_'
    else:
        dataset_name += 'noisy_'
    if mps_simulation:
        dataset_name += f'mps_chi={max_bond_dimension}_'
else:
    dataset_name += 'QPU_'

if time_block_size != 1.0:
    dataset_name += f'k={time_block_size}_'

if hamiltonian_model == HamiltonianModels.ErdosRenyi:
    dataset_name += f"ER-{instance_kwargs['p_or_M']}_"
elif hamiltonian_model == HamiltonianModels.RegularGraph:
    dataset_name += f"RG-{instance_kwargs['graph_degree']}_"
else:
    raise NotImplementedError(f"Hamiltonian model {hamiltonian_model} is not implemented.")

if no_gauges_version:
    dataset_name += 'no-gauges-version_'

if add_hdls:
    dataset_name += 'HDLS_'

dataset_name += RUN_ID

_existing_ids = read_dataset_info(dataset_name=dataset_name)
if len(_existing_ids) > 0:
    raise ValueError(f"dataset {dataset_name} already holds {len(_existing_ids)} "
                     f"experiment sets; choose a new RUN_ID")

ndar_metadata = {'local_samplers': [x.value for x in local_samplers],
                 'convergence_criterion': convergence_criterion.get_description_string(),
                 }

print('DATASET NAME:')
print(dataset_name)

## Run

One experiment set per (restart, instance). The coupling map is resolved per set with
`coupling_map='auto'`, which calls the library's calibration heuristic
(`filter_couplings_map_heuristic`) and drops the couplings the device reports as the
noisiest that day. The same procedure runs in both ansatz arms. It is not pinned across
days by design: the point of the comparison is that each arm meets the device as it is,
and a paired ND / Standard study should therefore be run on the same day.

In [ ]:
all_experiment_set_ids = {i: [] for i in hamiltonian_indices_range}

for seed_restart in restarts_range:
    #note: seed_restart matters only for simulations
    anf.wait_unless_interrupted(5)
    for hamiltonian_instance_index in hamiltonian_indices_range:
        anf.wait_unless_interrupted(1)

        cm_filtered = 'auto'
        pass_manager_kwargs = pass_manager_kwargs_main.copy()
        pass_manager_kwargs['coupling_map'] = cm_filtered

        qiskit_config_dict = qiskit_config_dict_main.copy()
        qiskit_config_dict['pass_manager_kwargs'] = pass_manager_kwargs

        qiskit_metadata_additional = qiskit_metadata_additional_main.copy()
        qiskit_metadata_additional['pass_manager_kwargs'] = pass_manager_kwargs

        config_qiskit_sampler = config_qiskit_sampler_main.copy()

        if simulation:
            qiskit_config_dict['qiskit_sampler_options'][
                'seed_simulator'] = 1000 * hamiltonian_instance_index + seed_restart

        config_qiskit_sampler['qiskit_config'] = qiskit_config_dict

        cost_hamiltonian_i: ClassicalHamiltonian = generate_random_hamiltonian(hamiltonian_model=hamiltonian_model,
                                                                               number_of_qubits=number_of_qubits,
                                                                               seed=hamiltonian_instance_index,
                                                                               default_backend=default_backend,
                                                                               localities=localities,
                                                                               instance_kwargs=instance_kwargs)
        #The analysis notebooks read both extremal energies of the instance from the store; a re-run reads them instead of solving again.
        if cost_hamiltonian_i.lowest_energy is None or cost_hamiltonian_i.highest_energy is None:
            cost_hamiltonian_i.solve_hamiltonian(both_directions=True)
        cost_hamiltonian_i.write_to_file()

        ham_clas_description = cost_hamiltonian_i.hamiltonian_class_description
        ham_instance_description = cost_hamiltonian_i.hamiltonian_instance_description

        hamiltonian_instance_metadata = cost_hamiltonian_i.hamiltonian_metadata
        cost_hamiltonian_i._spectrum = None

        anf.cool_print('RUNNING NEW HAMILTONIAN!', '...', 'red')

        experiment_set_id = f"{anf.create_random_uuid()}"
        all_experiment_set_ids[hamiltonian_instance_index].append(experiment_set_id)

        universal_logger_callable = build_universal_logging_callable()

        logger_kwargs_main = {'experiment_folders_hierarchy': experiments_folders_hierarchy,
                              'experiment_set_id': experiment_set_id,  #Used to group the experiments
                              'dataset_name': dataset_name,
                              }

        experiment_metadata = {"timestamp_start": datetime.now().isoformat(),
                               "quapopt_version": getattr(quapopt, '__version__', 'unknown'),
                               "python_version": sys.version.split()[0], }

        metadata_experiment_set = {"LoggerMetadata": logger_kwargs_main,
                                   "NDARMetadata": ndar_metadata,
                                   'ClassicalOptimizerMetadataQAOA': optimizer_metadata,
                                   'HamiltonianMetadata': hamiltonian_instance_metadata,
                                   'ExperimentMetadata': experiment_metadata,
                                   'ImplementationMetadataAdditional': qiskit_metadata_additional_main,
                                   'ImplementationMetadata': config_qiskit_sampler,
                                   "NoGaugesVersion": no_gauges_version,
                                   }

        with bck_utils.create_qiskit_session(qiskit_backend=qiskit_backend,
                                             mocked=simulation,
                                             mode='batch') as session_ibm:

            config_qiskit_sampler['qiskit_config']['session_ibm'] = session_ibm

            sampler_qiskit, updater_qiskit = build_local_sampler_callable_and_updater_NDAR(
                local_sampler_name=LocalSamplerName.QiskitWithOfflineAnglesP1,
                **config_qiskit_sampler,
                verbosity=0,
                logger_kwargs_main=logger_kwargs_main)

            metadata_experiment_set[
                f"{LocalSamplerName.QiskitWithOfflineAnglesP1.value}Metadata"] = config_qiskit_sampler
            metadata_experiment_set[
                f"{LocalSamplerName.QiskitWithOfflineAnglesP1.value}MetadataAdditional"] = qiskit_metadata_additional

            local_sampler_functions = [(sampler_qiskit,
                                        updater_qiskit,
                                        universal_logger_callable,
                                        LocalSamplerName.QiskitWithOfflineAnglesP1.value)]

            metadata_logger = ResultsLogger(**logger_kwargs_main,
                                            experiment_instance_id='nan')
            try:
                metadata_logger.write_metadata(metadata=metadata_experiment_set,
                                               data_type=SNDT.CircuitsMetadata,
                                               shared_across_experiment_set=True)
            except FileExistsError:
                anf.cool_print("WARNING:",
                               "The experiment set already exists. The results will be appended to the existing files.",
                               'red')

            df_metadata_simple_i = pd.DataFrame(data={SNV.ExperimentSetID.id_long: [experiment_set_id],
                                                      SNV.NumberOfQubits.id_long: [number_of_qubits],
                                                      SNV.HamiltonianClassDescription.id_long: [ham_clas_description],
                                                      SNV.HamiltonianInstanceDescription.id_long: [
                                                          ham_instance_description],
                                                      'rBestSolutions': [r_best_results],
                                                      'TranspilerSeeds': [pass_manager_seeds_list],
                                                      SNV.TimeBlockSize.id_long: [time_block_size],
                                                      f"{SNV.TimeBlock.id_long}{SNV.Seed.id_long}": [
                                                          time_block_seed_schedule],
                                                      'NumberOfSamples': [number_of_samples_qiskit],
                                                      'LocalSamplers': [
                                                          LocalSamplerName.QiskitWithOfflineAnglesP1.value],
                                                      'CouplingMapIsFiltered': [cm_filtered is not None],
                                                      })

            metadata_logger.write_results(dataframe=df_metadata_simple_i,
                                          data_type=SNDT.QAOAOptimizationMetadata)

            ndar_runner = NDARRunner(input_hamiltonian_representation=cost_hamiltonian_i,
                                     convergence_criterion=convergence_criterion)

            anf.cool_print("EXPERIMENT SET ID:", experiment_set_id, 'yellow')

            best_res_ndar, history_ndar = ndar_runner.run_NDAR(
                local_sampler_functions=local_sampler_functions,
                show_progress_bar_ndar=True,
                verbosity=1,
                optimize_over_r_gauges=r_best_results,
                break_after_finding_ground_state=simulation and not mps_simulation)

print('done:', dataset_name)